# Benchmark — Detección de vehículos y estimación de distancias
Mide el tiempo de procesamiento y consumo de recursos del pipeline completo (YOLO + estimación de distancias) para comparar PC vs Raspberry Pi.

In [1]:
import sys
import os
import time
import cv2
import numpy as np
import psutil
import statistics
import platform
import torch
import pynvml


# Ir a la raíz del proyecto para que los paths relativos de las clases funcionen
ruta_proyecto = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(ruta_proyecto)
sys.path.insert(0, os.path.join(ruta_proyecto, 'src'))

from detection.yolo_detection import YoloDetection
from distanceEstimation.Distance_Estimation import DistanceEstimation

# GPU monitoring — intenta pynvml, si falla usa torch.cuda
GPU_DISPONIBLE = False
GPU_MODO       = None
gpu_handle     = None
gpu_name       = "N/A"

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    try:
        pynvml.nvmlInit()
        gpu_handle     = pynvml.nvmlDeviceGetHandleByIndex(0)
        GPU_DISPONIBLE = True
        GPU_MODO       = "pynvml"
    except Exception:
        print("Error Inicializando pynvml")        


if GPU_DISPONIBLE:
    print(f"GPU detectada ({GPU_MODO}) : {gpu_name}")
else:
    print("GPU no disponible — solo se mide CPU/RAM")

print(f"Plataforma : {platform.machine()} — {platform.system()} {platform.release()}")
print(f"Python     : {platform.python_version()}")
print(f"Directorio : {os.getcwd()}")

GPU detectada (pynvml) : NVIDIA GeForce RTX 4070 Laptop GPU
Plataforma : AMD64 — Windows 10
Python     : 3.11.9
Directorio : c:\Users\nicoc\Downloads\Tesis NNs\TesisLab\Deteccion-de-vehiculos-y-estimacion-de-distancias-con-YOLOv8


In [2]:
# ── Cargar modelo y video ──────────────────────────────────────────────────
rutaModelo = os.path.join(ruta_proyecto, 'models', 'yolov8n.pt')
rutaVideo  = os.path.join(ruta_proyecto, 'data', 'samples', 'Evaluation.mp4')

detector = YoloDetection(rutaModelo)

cap = cv2.VideoCapture(rutaVideo)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps_video    = cap.get(cv2.CAP_PROP_FPS)
print(f"Video: {total_frames} frames a {fps_video:.1f} FPS")

# Warm-up: 20 inferencias previas para estabilizar YOLO antes de medir
N_WARMUP = 20
for _ in range(N_WARMUP):
    ret, frame_warmup = cap.read()
    if ret:
        detector.detectAndParse(frame_warmup)
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
print(f"Warm-up completado ({N_WARMUP} frames)")

Video: 1524 frames a 30.0 FPS

0: 384x640 1 person, 6 cars, 59.2ms
Speed: 19.6ms preprocess, 59.2ms inference, 9.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 7 cars, 8.8ms
Speed: 1.9ms preprocess, 8.8ms inference, 2.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 6 cars, 1 motorcycle, 9.2ms
Speed: 1.6ms preprocess, 9.2ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 6 cars, 1 motorcycle, 11.5ms
Speed: 2.1ms preprocess, 11.5ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 7 cars, 1 motorcycle, 8.9ms
Speed: 1.8ms preprocess, 8.9ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 7 cars, 1 motorcycle, 10.1ms
Speed: 2.6ms preprocess, 10.1ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 6 cars, 1 motorcycle, 8.2ms
Speed: 2.1ms preprocess, 8.2ms inference, 2.2ms postprocess per i

In [3]:
# ── Loop de benchmark ─────────────────────────────────────────────────────
tiempos_deteccion = []
tiempos_distancia = []
tiempos_umbral    = []
tiempos_total     = []
uso_cpu           = []
uso_ram_mb        = []
uso_gpu_pct       = []
uso_gpu_mem_mb    = []

proceso = psutil.Process(os.getpid())
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    t_inicio = time.perf_counter()

    t1 = time.perf_counter()
    detecciones = detector.detectAndParse(frame)
    t2 = time.perf_counter()

    z_ref, obj_min, distancias_vec = None, None, []
    try:
        z_ref, obj_min, distancias_vec = DistanceEstimation.distanciasIntervehiculares(detecciones)
    except Exception:
        pass
    t3 = time.perf_counter()

    try:
        if z_ref is not None:
            DistanceEstimation.clasificacionDeDistancia(z_ref)
        for d_AB, _ in distancias_vec:
            DistanceEstimation.clasificacionDeDistancia(d_AB)
    except Exception:
        pass
    t4 = time.perf_counter()

    tiempos_deteccion.append(t2 - t1)
    tiempos_distancia.append(t3 - t2)
    tiempos_umbral.append(t4 - t3)
    tiempos_total.append(t4 - t_inicio)
    uso_cpu.append(psutil.cpu_percent(interval=None))
    uso_ram_mb.append(proceso.memory_info().rss / 1024**2)

    if GPU_DISPONIBLE:
        if GPU_MODO == "pynvml":
            uso_gpu_pct.append(pynvml.nvmlDeviceGetUtilizationRates(gpu_handle).gpu)
            uso_gpu_mem_mb.append(pynvml.nvmlDeviceGetMemoryInfo(gpu_handle).used / 1024**2)
        else:
            uso_gpu_pct.append(torch.cuda.utilization(0))
            uso_gpu_mem_mb.append(torch.cuda.memory_allocated(0) / 1024**2)

cap.release()
print(f"Frames procesados: {len(tiempos_total)}")


0: 384x640 1 person, 6 cars, 12.3ms
Speed: 2.3ms preprocess, 12.3ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)
Vehiculo más cercano
5.5087600250832995 {'clase_id': 2, 'H_px': 455.7175598144531, 'H_mean': 1.55, 'bbox': (0.70513916015625, 380.9544982910156, 555.76025390625, 836.6720581054688), 'conf': 0.8308402299880981}

0: 384x640 1 person, 7 cars, 10.9ms
Speed: 1.8ms preprocess, 10.9ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)
Vehiculo más cercano
5.536660987477307 {'clase_id': 2, 'H_px': 453.4998474121094, 'H_mean': 1.55, 'bbox': (0.778106689453125, 378.6927795410156, 554.297119140625, 832.192626953125), 'conf': 0.8296365737915039}

0: 384x640 1 person, 6 cars, 1 motorcycle, 8.3ms
Speed: 1.8ms preprocess, 8.3ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
Vehiculo más cercano
2.3502626075960547 {'clase_id': 3, 'H_px': 572.7995300292969, 'H_mean': 0.8, 'bbox': (1555.66796875, 362.4780578613281, 1919.932373046875, 935.27

In [4]:
# ── Resultados ────────────────────────────────────────────────────────────
# Cambiá este valor según dónde corrás el notebook
PLATAFORMA = "PC"   # "PC" o "Raspberry Pi"

def resumen(nombre, datos_s):
    datos_ms = np.array(datos_s) * 1000
    print(f"  {nombre}")
    print(f"    Media   : {datos_ms.mean():.3f} ms")
    print(f"    Mediana : {np.median(datos_ms):.3f} ms")
    print(f"    P95     : {np.percentile(datos_ms, 95):.3f} ms")
    print(f"    P99     : {np.percentile(datos_ms, 99):.3f} ms")
    print(f"    Mín     : {datos_ms.min():.3f} ms")
    print(f"    Máx     : {datos_ms.max():.3f} ms")

n = len(tiempos_total)
fps_real = 1 / np.mean(tiempos_total)

print(f"========== BENCHMARK — {PLATAFORMA} ==========")
print(f"Frames procesados : {n}")
print()
resumen("1. Detección YOLO",            tiempos_deteccion)
print()
resumen("2. Estimación de distancias",   tiempos_distancia)
print()
resumen("3. Clasificación por umbrales", tiempos_umbral)
print()
resumen("Pipeline completo (1+2+3)",     tiempos_total)
print(f"    FPS estimados : {fps_real:.2f}")
print()
print(f"  CPU / RAM")
print(f"    CPU media : {np.mean(uso_cpu):.1f}%")
print(f"    RAM media : {np.mean(uso_ram_mb):.1f} MB")
print(f"    RAM pico  : {np.max(uso_ram_mb):.1f} MB")

if GPU_DISPONIBLE and uso_gpu_pct:
    print()
    print(f"  GPU ({gpu_name})")
    print(f"    Utilización media : {np.mean(uso_gpu_pct):.1f}%")
    print(f"    Utilización pico  : {np.max(uso_gpu_pct):.1f}%")
    print(f"    VRAM media        : {np.mean(uso_gpu_mem_mb):.1f} MB")
    print(f"    VRAM pico         : {np.max(uso_gpu_mem_mb):.1f} MB")

========== BENCHMARK — PC ==========
Frames procesados : 1524

  1. Detección YOLO
    Media   : 15.195 ms
    Mediana : 14.550 ms
    P95     : 19.752 ms
    P99     : 24.746 ms
    Mín     : 11.494 ms
    Máx     : 123.584 ms

  2. Estimación de distancias
    Media   : 0.839 ms
    Mediana : 0.776 ms
    P95     : 1.295 ms
    P99     : 1.628 ms
    Mín     : 0.528 ms
    Máx     : 2.968 ms

  3. Clasificación por umbrales
    Media   : 0.002 ms
    Mediana : 0.002 ms
    P95     : 0.004 ms
    P99     : 0.006 ms
    Mín     : 0.001 ms
    Máx     : 0.012 ms

  Pipeline completo (1+2+3)
    Media   : 16.037 ms
    Mediana : 15.372 ms
    P95     : 20.716 ms
    P99     : 26.015 ms
    Mín     : 12.121 ms
    Máx     : 124.211 ms
    FPS estimados : 62.36

  CPU / RAM
    CPU media : 39.4%
    RAM media : 1479.9 MB
    RAM pico  : 1485.4 MB

  GPU (NVIDIA GeForce RTX 4070 Laptop GPU)
    Utilización media : 30.1%
    Utilización pico  : 52.0%
    VRAM media        : 1943.9 MB
    VRA